# Public notebook copy

Published from the author's course-project archive. Student identifiers and private paths have been removed; paths now use bundled repository data. Known attribution, numbering, distribution-framing and stale summary errors were corrected. The original private files are unchanged.

Code outputs are cleared so you can run the notebook from top to bottom.

The GPU inference notebook is still missing. These notebooks document the project or analyze precomputed scores. See the repository README for current results and scope.


# F-chelsea-zhao-0: Documentation Notebook

**MIE1512 Course Project — Final Submission**
Author: Chelsea Zhao
Paper reproduced: Xia et al., "Evaluating Mathematical Reasoning Beyond Accuracy," *AAAI 2025* (arXiv:2404.05692)

## Reproducibility on Google Colab

This submission is structured to run **end-to-end on Google Colab (CPU-only)** from pre-computed ReasonEval inference outputs. The GPU inference step was performed offline on a local RTX 5070 and its results are cached in `F-data/`.

### Folder layout (on Google Drive)
```
My Drive/MIE1512 Project/chelsea-zhao/
├── F-data/
│   ├── processbench_cached.parquet          # EDA input (full ProcessBench, 3,400 rows)
│   ├── processbench_sample.parquet          # 570-row stratified sample (V1 carry-over)
│   ├── processbench_full_results.parquet    # ReasonEval scores, full 3,400 rows
│   └── processbench_08_results.csv
└── F-notebooks/
    ├── F-chelsea-zhao-0.ipynb    (this notebook — documentation)
    ├── F-chelsea-zhao-1.ipynb    (data preparation + EDA)
    └── F-chelsea-zhao-2.ipynb    (analysis — reproduces every figure and table)
```

### How to reproduce
1. Upload (or place) the `chelsea-zhao/` folder to Google Drive so the path above exists.
2. Open `F-chelsea-zhao-1.ipynb` in Colab (CPU runtime is sufficient; no GPU needed).
3. In the first code cell, confirm/adjust `BASE_PATH` to match your Drive location — this is the **only** line a grader needs to edit.
4. Run the notebook top-to-bottom (Runtime → Run all).
5. Repeat for `F-chelsea-zhao-2.ipynb`.

### Why the old GPU-inference notebook is not included
The original V1/Final pipeline had a third working notebook (`F-chelsea-zhao-2` in earlier drafts) that ran ReasonEval-Mistral-7B inference on all 3,400 ProcessBench rows. That step requires a CUDA-13/Blackwell nightly PyTorch build, an 8 GB VRAM GPU, a local ReasonEval-7B checkpoint, and roughly 2 hours of wall-clock time — none of which are grader-reproducible in Colab's free CPU tier. Its outputs are frozen in `F-data/processbench_full_results.parquet` and consumed directly by Notebook 2.

**Final run order:** `0 → 1 → 2`.

## Table of Contents

1. [Project Plan](#1-project-plan)
2. [Time Plan](#2-time-plan)
3. [Notebook Inventory](#3-notebook-inventory)
4. [References](#4-references)

## Time Reporting

| Submission | Actual Hours | Status |
|---|---|---|
| V1 | 10.0 h | Complete — frozen in `V1-notebooks/` |
| Final | 20.0 h | Complete — this submission |
| **Total** | **30.0 h** | |

See Section 2 below for the per-task breakdown.

---

## 1. Project Plan

### Summary

This project evaluates **ReasonEval-Mistral-7B** (Xia et al., AAAI 2025 [1]) as a step-level
mathematical reasoning evaluator against **ProcessBench** (Zheng et al., ICLR 2025 [2]),
a human-annotated benchmark of 3,400 solutions across four difficulty levels (GSM8K, MATH,
OlympiadBench, OmniMath). ReasonEval produces per-step validity and redundancy scores; only
the **validity** dimension is evaluated here, as ProcessBench provides first-error-step
annotations but no redundancy labels.

The V1 submission scored a 570-row stratified sample as a viability check. The Final
submission runs ReasonEval-Mistral-7B on the **full 3,400-row ProcessBench dataset** and
re-generates all figures, tables, and metrics at full scale.

---

### Research Question

> Is ReasonEval-Mistral-7B a reliable proxy for human judgment on step-level reasoning quality
> when evaluated against ProcessBench — a benchmark spanning four difficulty levels with
> human-annotated step-level labels — compared to the original paper's evaluation suite
> (PRM800K for training; MR-MATH for in-distribution evaluation; MR-GSM8K for out-of-distribution evaluation)?
> Specifically, does ReasonEval's claimed out-of-distribution generalization hold under stricter human
> annotation on harder subsets (OlympiadBench, OmniMath) that exceed the difficulty of its
> PRM800K training data?

---

### Dataset Pipeline

| | PRM800K | MR-MATH | MR-GSM8K | ProcessBench |
|--|---------|---------|----------|--------------|
| **Role** | Training | In-distribution eval | Out-of-distribution eval | **This project** |
| **Source** | MATH | MATH | GSM8K | GSM8K · MATH · OlympiadBench · OmniMath |
| **Difficulty** | Competition | Competition | Grade school | Grade school → Olympiad |
| **Solutions** | GPT-4 | Abel · WizardMath | MetaMath-7B | 12 open-source LLMs (1.5B–72B) |
| **Labels** | positive / neutral / negative per step | First error step | Correct / Wrong per step | First error step index |
| **Annotator** | Human (OpenAI) | Human (undergrad) | Human (dual annotation + QC supervisor) | Human (doctoral) |
| **Size** | 75K solutions · 800K step labels | 159 (83 incorrect · 76 correct) | ~1.4K per subset · ~2.8K total | 3,400 (400 GSM8K + 1,000 each for MATH, OlympiadBench, OmniMath) |

> **Note:** The original paper's out-of-distribution evaluation uses MR-GSM8K (grade school level — *easier* than
> training). ProcessBench extends this by also covering OlympiadBench and OmniMath
> (olympiad level — *harder* than training), testing out-of-distribution generalization in both directions.

---

### Paper Sections Reproduced

| Paper Section | Page | Target | Description |
|---|---|---|---|
| Section 1, Figure 1 | p.2 | Step scoring example | Qualitative per-step S_validity + S_redundancy with error location |
| Section 4.2, Tables 1 & 2 | p.4–5 | AUC + F1 Score | Solution- and step-level detection vs. ProcessBench human labels |
| Section 4.2, Figure 2 | p.5 | Pass@1 vs AUC | Correlation between generator final-answer accuracy and detection AUC |
| Section 5.1, Figure 3 | p.6 | Score distributions | S_validity + S_redundancy box plots by correct/incorrect, per generator family |
| Section 5.1, Table 4 | p.6 | False Positive Rate per generator | FPR at S_validity < 0.25 threshold per generator model |
| Section 5.1, Figure 4 | p.7 | Accuracy vs FPR scatter | ProcessBench generators overlaid on original paper models |

---

### Implementation Design

- **Evaluator:** ReasonEval-Mistral-7B — the 34B Llemma variant (best performing in the paper) is
  not feasible on 8 GB VRAM (NVIDIA RTX 5070, GPU+CPU offload via `device_map='auto'`).
- **Dataset:** ProcessBench — 3,400 human-annotated solutions, 12 generators (Llama + Qwen, 1.5B–72B).
- **Ground truth:** ProcessBench `label` field — 0-indexed first erroneous step, or −1 if correct.
- **V1 sample:** 570 stratified rows (6 per generator × subset × error group combination) — used as a
  viability check only; superseded by the full 3,400-row Final run.
- **Final run:** All 3,400 rows scored; inference took ≈ 2 hours of GPU time.
- **Environment:** VS Code, Python 3.12, virtual environment `venMIE`, Windows 11, RTX 5070 (8 GB VRAM).

### ReasonEval Score Thresholds (Xia et al., Appendix D/E, p.12)

| Threshold | Value | Used for |
|---|---|---|
| Validity — step/solution classification | S_validity < 0.5 | F1 Score (Tables 1 & 2) |
| Validity — False Positive Rate | S_validity < 0.25 | False Positive Rate (Table 4) |
| Redundancy — step classification | S_redundancy > 0.15 | Omitted — no redundancy labels in ProcessBench |

---

## 2. Time Plan

*Per Section 6 of the submission requirements. Half-hour granularity. Paper-reading time
excluded (allocated to Bibliography and Presentation per Section 3.1).*

### V1 Actual Hours (~10 h, frozen in V1-notebooks/)

| Task | Hours | Description |
|------|-------|-------------|
| Dependency engineering and environment configuration | 0.5 h | PyTorch nightly cu128 build for CUDA 13.0 / Blackwell (RTX 5070); resolved version conflicts |
| ReasonEval-7B model class implementation | 1.5 h | Studied official ReasonEval repo; adapted `MistralPreTrainedModel`; configured tokenizer, device mapping |
| Dataset engineering and selection | 2.5 h | Parsers and EDA for MathShepherd (444K rows), NuminaMath-CoT, OpenMathInstruct-1/2; ProcessBench schema validation |
| ProcessBench data pipeline | 1.5 h | Data loading with local parquet caching; step-count distributions, error rates, generator × subset heatmap |
| Stratified sampler implementation | 1.0 h | 570-row sample across 12 generators × 4 subsets × 2 error groups; fallback logic for missing groups |
| Inference pipeline engineering (sample) | 1.5 h | `device_map='auto'` offload; incremental checkpoint saving; sample-scale run |
| Metric computation implementation | 1.0 h | Ported `compute_metrics()` from official repo; macro F1, AUC, FPR; 0-indexed → 1-indexed label alignment |
| V1 figures and tables | 0.5 h | Figures 1–4 and Tables 1, 2, 4 on the 570-row sample |
| **V1 Total** | **10.0 h** | |

### Final Actual Hours (~20 h, this submission)

| Task | Hours | Paper Reference | Description |
|------|-------|-----------------|-------------|
| Scaling the ReasonEval-Mistral-7B pipeline from the 570-row V1 sample to the full 3,400-row ProcessBench run | 3.0 h | §3.1–3.3, §4.2 | Extended V1 inference pipeline with resume-from-checkpoint logic, incremental per-row persistence, and a skip-list for the 570 V1-scored IDs; revalidated parity with V1 on overlapping rows |
| Debugging and recovering from CUDA memory pressure during the full-dataset run | 1.5 h | §3.3 (device_map) | Diagnosed intermittent CUDA OOM on longer OlympiadBench solutions (60+ steps); tightened the device_map="auto" offload boundary; added explicit torch.cuda.empty_cache() between samples; re-ran affected shards |
| Full-scale metric computation (Tables 1 & 2, all 3,400 rows) | 1.5 h | §4.2, Tables 1–2, p.4–5 | Ported V1 compute_metrics() to the 3,400-row result set; F1 and AUC per subset (GSM8K, MATH, OlympiadBench, OmniMath) plus pooled; sanity-checked against V1 sample-scale numbers |
| Figure 1 regeneration on full dataset | 0.5 h | §1, Figure 1, p.2 | Rebuilt the qualitative step-card figure with candidate selection from the full pool; re-surfaced a representative MATH example |
| Figure 2 regeneration on full dataset | 1.0 h | §4.2, Figure 2, p.5 | Per-generator Pass@1 vs solution-level AUC bubble chart; recomputed per-generator accuracy and AUC on all 3,400 rows |
| Figure 3 regeneration on full dataset | 1.5 h | §5.1, Figure 3, p.6 | S_validity / S_redundancy distributions by generator and correctness; re-tuned family ordering, threshold lines, axis ranges |
| Table 4 and Figure 4 regeneration on full dataset | 1.5 h | §5.1, Table 4 & Figure 4 | FPR per generator at S_validity < 0.25; merged with paper-model overlay scatter; adjusted marker jitter for zero-FPR points |
| Investigation of the "Know What You Don't Know" (KWDK) uncertainty-calibration methodology [8] | 3.0 h | Park et al. 2025 [8] | Read the KWDK paper on PRM uncertainty calibration; prototyped a calibration pass over ReasonEval step-validity scores (reliability diagrams, expected calibration error); integration with the ReasonEval logit layout required non-trivial re-plumbing of the scoring function, so the feature was cut from the Final scope |
| Exploratory threshold sensitivity analysis | 1.0 h | Appendix D, p.12 | Swept the S_validity threshold from 0.1 to 0.9 on a held-out subset to check whether the paper's 0.5 / 0.25 defaults are near-optimal for ProcessBench; confirmed directionality and scoped the analysis out of the Final report because it did not change the headline conclusions |
| Migration of inference output from CSV to Parquet and rewiring Notebook N for Parquet-only inputs | 0.5 h | §3.2, §3.3 | Satisfies the submission's "Parquet inputs only for Notebook N" rule; Snappy-compressed Parquet output, all downstream loads re-pointed |
| Rewriting notebook markdown from V1 to Final voice; re-linking F-0 / F-1 / F-2 / F-3 chain | 1.0 h | §3.1, §3.2 | Reworded section headers, rewrote the notebook-inventory entries, and verified cross-notebook filename references after the V1→F rename |
| Final report writing (IEEE style, 5–10 pages, single column, 11 pt) | 3.0 h | All sections | Drafted the abstract, introduction, dataset summary, methods, results (Figures 1–4, Tables 1–2), discussion, and bibliography; kept all content consistent with the outputs of Notebook N |
| Notebook cleanup and end-to-end reproducibility validation (Runtime > Run All on each of F-0 to F-3) | 1.0 h | Section 8 | Walked each F-notebook top to bottom, confirmed every cell executes in order, verified no code cell is missing the markdown explanation above it |
| **Final Total** | **20.0 h** | | |
| **Grand Total** | **30.0 h** | | |

> **Notes on excluded time:**
> - Paper-reading time for the primary paper (Xia et al.) is excluded per Section 3.1 (allocated to Bibliography and Presentation).
> - GPU compute time (~2 h wall clock for the full ReasonEval inference on 3,400 rows) is excluded per course guidelines; the hours above reflect engineering, debugging, and analysis work only.

---

## 3. Notebook Inventory

*Per Section 3.1 of the submission requirements. All four Final notebooks live in `F-notebooks/`.
All data products live in `F-data/` (reusing `F-data/processbench_cached.parquet` unchanged, per Section 2.2).*

---

### F-chelsea-zhao-1.ipynb — Data Preparation (intermediate, 1 of N−1)

**Purpose:** ProcessBench data loading, EDA, stratified sample construction.
Produces the Parquet files consumed by the downstream Final notebooks.

**Data files consumed:** HuggingFace `Qwen/ProcessBench` (first run) or local Parquet cache.

**Data files produced:**
- `F-data/processbench_cached.parquet` — full ProcessBench dataset (3,400 rows) *(reused from V1 per Section 2.2)*
- `F-data/processbench_sample.parquet` — 570-row stratified sample *(reused from V1)*

---

### F-chelsea-zhao-2.ipynb — Full-Dataset Inference (intermediate, N−1)

**Purpose:** Run ReasonEval-Mistral-7B inference on all 3,400 ProcessBench rows. Resume-capable
from partial checkpoints. Writes results as Parquet for consumption by Notebook N.

**Code cells (high level):** load model, define `score_solution()`, iterate the full dataset
with incremental checkpointing, save results as Parquet.

**Data files consumed:**
- `F-data/processbench_cached.parquet` — full ProcessBench dataset
- `F-data/processbench_08_results.csv` — 570 V1 sample results (pre-populated into the Final output to avoid re-scoring)

**Data files produced:**
- `F-data/processbench_full_results.parquet` — ReasonEval scores for all 3,400 solutions

---

### F-chelsea-zhao-3.ipynb — Final Analysis (Notebook N)

**Purpose:** Reproducible, end-to-end analysis on the full 3,400-row ProcessBench dataset.
Consumes **only Parquet files**. Re-generates every figure and table from the paper.

**Data files consumed (Parquet only):**
- `F-data/processbench_cached.parquet` — ProcessBench labels (for alignment)
- `F-data/processbench_full_results.parquet` — ReasonEval scores from F-2

**Data files produced (all in `F-data/`):**
- `F-data/full_figure1_step_example_cards.png`
- `F-data/full_figure2_acc_vs_auc.png`
- `F-data/full_figure3_score_distributions.png`
- `F-data/full_figure4_acc_vs_fpr.png`
- `F-data/full_table4_fpr.csv`

---

## 4. References

*The first reference is the paper being reproduced, per Section 3.3 of the submission requirements.
References are grouped by their actual role in the project: primary reproduction sources first;
datasets that were explored during V1 but not used in the Final run; and methodologies investigated
during the Final phase but scoped out.*

### 4.1 Primary references used in the Final reproduction

1. Xia, S., Li, X., Liu, Y., Wu, T., and Liu, P.
   "Evaluating Mathematical Reasoning Beyond Accuracy." *Proceedings of the AAAI Conference on
   Artificial Intelligence (AAAI 2025).* arXiv:2404.05692.
   — Paper being reproduced. Sections 3.1–3.3 (scoring), 4.2 (Tables 1–2, Figure 2),
     5.1 (Figure 3, Table 4, Figure 4), Appendix D–E (thresholds).

2. Zheng, C., et al. "ProcessBench: Identifying Process Errors in Mathematical Reasoning."
   *Annual Meeting of the Association for Computational Linguistics (ACL 2025).* arXiv:2412.06559.
   — Replacement evaluation dataset (3,400 human-annotated solutions across GSM8K, MATH,
     OlympiadBench, OmniMath).

3. Lightman, H., Kosaraju, V., Burda, Y., Edwards, H., Baker, B., Lee, T., Leike, J.,
   Schulman, J., Sutskever, I., and Cobbe, K. "Let's Verify Step by Step." OpenAI
   (PRM800K dataset), 2023. Available: https://github.com/openai/prm800k.
   — Training dataset for ReasonEval; referenced in methods for the scoring convention
     and the in-distribution / out-of-distribution framing.

4. GAIR-NLP. "ReasonEval." GitHub repository, 2024.
   Available: https://github.com/GAIR-NLP/ReasonEval.
   — Source of the official scoring script and compute_metrics() implementation ported
     into F-chelsea-zhao-2 and F-chelsea-zhao-3.

### 4.2 Datasets evaluated during V1 exploration (not used in the Final reproduction)

*These were surveyed during V1 dataset engineering (parsers, EDA, schema validation) as
candidate replacement datasets before ProcessBench was selected. They do not appear in the
Final analysis pipeline.*

5. Wang, P., et al. "Math-Shepherd: Verify and Reinforce LLMs Step-by-step without Human
   Annotations." *Proc. 62nd ACL*, Bangkok, Thailand, 2024, pp. 9426–9439.
   DOI: 10.18653/v1/2024.acl-long.510.
   — 444K-row process-reward dataset surveyed during V1; rejected because of its
     28.2 % non-monotonic label rate on step-level rewards.

6. Numina. "NuminaMath-CoT." HuggingFace, 2024.
   Available: https://huggingface.co/datasets/AI-MO/NuminaMath-CoT.
   — Candidate training-style dataset reviewed during V1; lacks human step-level error
     annotations required for this project.

7. Toshniwal, S., et al. "OpenMathInstruct-1: A 1.8 Million Math Instruction Tuning Dataset."
   arXiv:2402.10176, 2024.
   — Instruction-tuning dataset reviewed during V1 dataset engineering; not selected.

8. Toshniwal, S., et al. "OpenMathInstruct-2: Accelerating AI for Math with Massive
   Open-Source Instruction Data." arXiv:2410.01560, 2024.
   — Successor to OpenMathInstruct-1; also reviewed during V1 dataset engineering.

### 4.3 Methodology investigated during the Final phase (not integrated)

9. Park, Y.-J., Greenewald, K., Alim, K., Wang, H., and Azizan, N. "Know What You Don't Know: Uncertainty Calibration of Process
   Reward Models." arXiv pre-print, 2025.
   — KWDK calibration methodology for PRM uncertainty. Read and prototyped during the
     Final phase as a possible extension of ReasonEval's step-validity output (reliability
     diagrams, expected calibration error); integration with the ReasonEval three-logit
     head required non-trivial re-plumbing of the scoring function and was scoped out
     of the Final deliverable.